# Tutorial 21: Functional API in LangGraph

In this tutorial we explore LangGraph's Functional API — an alternative to `StateGraph` that lets you write workflows as plain Python functions using two decorators: `@entrypoint` and `@task`. It shares the same runtime, the same checkpointing, and the same HITL primitives as `StateGraph`.

## 1. StateGraph vs Functional API

| | StateGraph | Functional API |
|---|---|---|
| Structure | Explicit nodes + edges | Plain Python functions |
| Control flow | `add_edge`, `add_conditional_edges` | `if/else`, `for`, `while` |
| Checkpointing | After every node | After every `@task` |
| Visualisation | `get_graph().draw_mermaid_png()` | Limited |
| Best for | Complex branching, parallelism | Sequential, iterative workflows |

Both APIs run on the same underlying runtime and can be mixed — a `@task` can invoke a compiled `StateGraph` and vice-versa.

## 2. Setup

In [1]:
import os
from langgraph.func import entrypoint, task
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0.1,
)

print("Setup complete.")

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Setup complete.


## 3. Core Decorators

### `@task`
Marks a function as a discrete unit of work. Its result is persisted as a checkpoint. If execution is interrupted and resumed, completed tasks are **not re-run**.

### `@entrypoint`
Marks the workflow entry point. Accepts a single input, manages checkpointing, and exposes:
- `previous` — the return value from the last invocation on the same thread
- `store` — the cross-thread memory store (if one is configured)

In [2]:
@task
def fetch_context(topic: str) -> str:
    """Simulate fetching context for a topic."""
    # In production this would call a search API or database
    return f"Context about {topic}: This is a rapidly evolving field with many applications."


@task
def generate_summary(context: str) -> str:
    """Generate a summary from context."""
    resp = llm.invoke([HumanMessage(content=f"Summarise this in one sentence:\n{context}")])
    return resp.content


@task
def generate_questions(summary: str) -> list:
    """Generate follow-up questions from a summary."""
    resp = llm.invoke([HumanMessage(content=f"Generate 3 follow-up questions about:\n{summary}")])
    return resp.content.split("\n")


checkpointer = MemorySaver()


@entrypoint(checkpointer=checkpointer)
def research_workflow(topic: str) -> dict:
    """Workflow: fetch → summarise → generate questions."""
    context = fetch_context(topic).result()
    summary = generate_summary(context).result()
    questions = generate_questions(summary).result()
    return {"summary": summary, "questions": questions}


print("Functional workflow defined.")

Functional workflow defined.


In [3]:
config = {"configurable": {"thread_id": "functional-run-1"}}

result = research_workflow.invoke("quantum computing", config)
print("Summary:", result["summary"])
print("\nFollow-up questions:")
for q in result["questions"][:3]:
    if q.strip():
        print(f"  - {q.strip()}")

Summary: Quantum computing is a rapidly evolving field with numerous potential applications.

Follow-up questions:
  - Here are three follow-up questions about quantum computing:
  - 1. **What are some of the current challenges in scaling up quantum computing systems to make them more practical and accessible for widespread use?**


## 4. The `previous` Parameter — Stateful Iterations

When you invoke the same `thread_id` again, `previous` receives the return value from the last invocation. This enables iterative refinement without any explicit state management.

In [4]:
@task
def refine_text(text: str, feedback: str) -> str:
    resp = llm.invoke([HumanMessage(content=f"Improve this text based on the feedback.\nText: {text}\nFeedback: {feedback}")])
    return resp.content


iterative_checkpointer = MemorySaver()


@entrypoint(checkpointer=iterative_checkpointer)
def iterative_writer(input: dict, *, previous: dict = None) -> dict:
    """Each invocation refines the previous version using new feedback."""
    topic = input.get("topic", "")
    feedback = input.get("feedback", "")

    if previous is None:
        # First run: generate initial draft
        resp = llm.invoke([HumanMessage(content=f"Write a short paragraph about: {topic}")])
        text = resp.content
        iteration = 1
    else:
        # Subsequent runs: refine using feedback
        text = refine_text(previous["text"], feedback).result()
        iteration = previous["iteration"] + 1

    return {"text": text, "iteration": iteration}


iter_config = {"configurable": {"thread_id": "iterative-writer-1"}}

# First draft
v1 = iterative_writer.invoke({"topic": "the importance of sleep"}, iter_config)
print(f"Iteration {v1['iteration']}:")
print(v1["text"][:200])

Iteration 1:
Sleep plays a vital role in maintaining our overall health and well-being. During sleep, our bodies repair and regenerate damaged cells, build bone and muscle, and strengthen our immune systems. Adequ


In [5]:
# Refine with feedback — previous automatically contains v1's result
v2 = iterative_writer.invoke({"feedback": "Add specific statistics and make it more engaging."}, iter_config)
print(f"Iteration {v2['iteration']}:")
print(v2["text"][:250])

Iteration 2:
**The Power of Sleep: Unlocking a Healthier, Happier You**

Sleep is the unsung hero of our daily lives, playing a vital role in maintaining our overall health and well-being. During those precious hours of slumber, our bodies undergo a remarkable tr


## 5. Human-in-the-Loop with the Functional API

`interrupt()` works identically in the Functional API — it pauses the `@entrypoint` and is resumed with `Command(resume=...)`.

In [6]:
hitl_checkpointer = MemorySaver()


@task
def draft_content(prompt: str) -> str:
    resp = llm.invoke([HumanMessage(content=prompt)])
    return resp.content


@entrypoint(checkpointer=hitl_checkpointer)
def content_pipeline(topic: str) -> dict:
    """Draft → human review → publish."""
    draft = draft_content(f"Write a short blog post intro about: {topic}").result()

    print("\n--- DRAFT FOR REVIEW ---")
    print(draft[:200])
    print("------------------------")

    # Pause for human approval
    decision = interrupt({"draft": draft, "question": "Approve this draft? (yes/revise)"})

    if decision.lower() != "yes":
        draft = draft_content(f"Rewrite this blog intro with feedback '{decision}':\n{draft}").result()

    return {"published": True, "content": draft}


hitl_config = {"configurable": {"thread_id": "content-pipeline-1"}}

# Run until interrupt
content_pipeline.invoke("the future of remote work", hitl_config)
print("\nGraph paused — waiting for review.")


--- DRAFT FOR REVIEW ---
**The Future of Remote Work: Trends, Challenges, and Opportunities**

As we continue to navigate the post-pandemic landscape, one thing is clear: remote work is here to stay. With the rise of digital 
------------------------

Graph paused — waiting for review.


In [7]:
# Resume with approval
result = content_pipeline.invoke(Command(resume="yes"), hitl_config)
print("Published:", result["published"])
print("Content:", result["content"][:200])


--- DRAFT FOR REVIEW ---
**The Future of Remote Work: Trends, Challenges, and Opportunities**

As we continue to navigate the post-pandemic landscape, one thing is clear: remote work is here to stay. With the rise of digital 
------------------------
Published: True
Content: **The Future of Remote Work: Trends, Challenges, and Opportunities**

As we continue to navigate the post-pandemic landscape, one thing is clear: remote work is here to stay. With the rise of digital 


## 6. Parallel Tasks

Call multiple `@task` functions before calling `.result()` to run them concurrently.

In [8]:
@task
def get_pros(topic: str) -> str:
    resp = llm.invoke([HumanMessage(content=f"List 3 advantages of: {topic}")])
    return resp.content


@task
def get_cons(topic: str) -> str:
    resp = llm.invoke([HumanMessage(content=f"List 3 disadvantages of: {topic}")])
    return resp.content


@task
def get_examples(topic: str) -> str:
    resp = llm.invoke([HumanMessage(content=f"Give 2 real-world examples of: {topic}")])
    return resp.content


parallel_checkpointer = MemorySaver()


@entrypoint(checkpointer=parallel_checkpointer)
def balanced_analysis(topic: str) -> dict:
    # Launch all three tasks — they run concurrently
    pros_future = get_pros(topic)
    cons_future = get_cons(topic)
    examples_future = get_examples(topic)

    # Collect results
    return {
        "pros": pros_future.result(),
        "cons": cons_future.result(),
        "examples": examples_future.result(),
    }


import time
start = time.time()
result = balanced_analysis.invoke("remote work", {"configurable": {"thread_id": "analysis-1"}})
elapsed = time.time() - start

print(f"Completed in {elapsed:.1f}s")
print("\nPros:", result["pros"][:100])
print("Cons:", result["cons"][:100])
print("Examples:", result["examples"][:100])

Completed in 4.5s

Pros: Here are three advantages of remote work:

1. **Increased Flexibility and Work-Life Balance**: Remot
Cons: Here are 3 disadvantages of remote work:

1. **Social Isolation and Lack of Human Interaction**: Rem
Examples: Here are 2 real-world examples of remote work:

1. **Dell's Remote Work Policy**: In 2020, Dell anno


## 7. Conclusion

In this tutorial we explored LangGraph's Functional API:
- `@task` defines a discrete work unit whose result is persisted as a checkpoint
- `@entrypoint` is the workflow entry point, supporting `previous` for stateful iterations
- `interrupt()` works identically to `StateGraph` — same `Command(resume=...)` pattern
- Launching multiple tasks before calling `.result()` runs them in parallel
- Both APIs share the same runtime and can be mixed freely

In Tutorial 22 we explore Deep Agents — the official LangChain abstraction for building agents with planning, sub-agent spawning, and filesystem access.